# 🔬 Autonomous AI Research Scientist Agent
### Powered by LangGraph + Groq (llama-3.1-8b-instant) + Serper API

**Workflow:** Research Problem → Literature Review → Hypothesis Generation → Experiments → Analysis → Paper

**Agents:** Literature Review | Hypothesis Generator | Experiment | Result Analysis | Paper Writer

**Tools:** arXiv API | Serper Web Search | scikit-learn ML Tools


## 📦 Cell 1: Install All Required Libraries


In [1]:
# ============================================================
# CELL 1: Install All Required Libraries
# ============================================================
print("Installing packages...")
import subprocess, sys

packages = [
    "langgraph", "langchain", "langchain-groq", "langchain-community",
    "gradio>=4.0.0", "arxiv", "requests", "httpx",
    "scikit-learn", "numpy", "pandas", "matplotlib", "seaborn",
    "scipy", "statsmodels", "python-dotenv", "typing-extensions", "pydantic",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("All packages installed!")


Installing packages...
All packages installed!


## 🔑 Cell 2: Import Libraries & Set API Keys


In [2]:
# ============================================================
# CELL 2: Import Libraries & Configure API Keys
# ============================================================
import os, json, time, re, random, warnings, traceback, io, base64
from datetime import datetime
from typing import TypedDict, List, Dict, Any, Optional

warnings.filterwarnings("ignore")

# LangChain / LangGraph
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END, START

# Data / ML
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression, load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, mean_squared_error, r2_score)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy import stats

# Web / UI
import requests
import arxiv
import gradio as gr
from IPython.display import display, HTML
from PIL import Image as PILImage

print("All libraries imported!")

# ============================================================
#  API KEY CONFIGURATION  -- Replace with your actual keys
# ============================================================
GROQ_API_KEY   = "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"    # https://console.groq.com
SERPER_API_KEY = "34cf9bb4a6965f4ad9bcce86f48f5d693048ac09"  # https://serper.dev

os.environ["GROQ_API_KEY"]   = GROQ_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

MODEL_NAME        = "llama-3.1-8b-instant"
MODEL_TEMPERATURE = 0.7
MODEL_MAX_TOKENS  = 2048

print(f"Model       : {MODEL_NAME}")
print(f"Temperature : {MODEL_TEMPERATURE}")
print(f"Max Tokens  : {MODEL_MAX_TOKENS}")
print("WARNING: Replace placeholder API keys before running!")


All libraries imported!
Model       : llama-3.1-8b-instant
Temperature : 0.7
Max Tokens  : 2048


## 🛠️ Cell 3: Initialize Groq LLM & Define Tools


In [3]:
# ============================================================
# CELL 3: Initialize Groq LLM & Define Tools
# ============================================================

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=MODEL_TEMPERATURE,
    max_tokens=MODEL_MAX_TOKENS,
    groq_api_key=GROQ_API_KEY,
)
print(f"Groq LLM initialized: {MODEL_NAME}")


# -- Tool 1: arXiv Search -------------------------------------------------
def search_arxiv(query: str, max_results: int = 5) -> str:
    try:
        client = arxiv.Client()
        search = arxiv.Search(query=query, max_results=max_results,
                              sort_by=arxiv.SortCriterion.Relevance)
        results = []
        for paper in client.results(search):
            results.append({
                "title"   : paper.title,
                "authors" : [a.name for a in paper.authors[:3]],
                "abstract": paper.summary[:300] + "...",
                "url"     : paper.entry_id,
                "year"    : paper.published.year if paper.published else "N/A",
            })
        return json.dumps(results, indent=2)
    except Exception as e:
        return f"arXiv error: {e}"


# -- Tool 2: Serper Web Search --------------------------------------------
def search_web(query: str, num_results: int = 5) -> str:
    try:
        url     = "https://google.serper.dev/search"
        headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
        payload = {"q": query, "num": num_results}
        resp    = requests.post(url, headers=headers, json=payload, timeout=10)
        data    = resp.json()
        results = [
            {"title": i.get("title",""), "snippet": i.get("snippet",""), "link": i.get("link","")}
            for i in data.get("organic", [])[:num_results]
        ]
        return json.dumps(results, indent=2)
    except Exception as e:
        return f"Web search error: {e}"


# -- Tool 3: ML Experiment ------------------------------------------------
def run_ml_experiment(experiment_type="classification",
                      dataset="synthetic",
                      model_name="random_forest") -> Dict[str, Any]:
    res: Dict[str, Any] = {"experiment_type": experiment_type,
                            "model": model_name, "dataset": dataset}
    try:
        # Load dataset
        if dataset == "iris":
            d = load_iris();          X, y = d.data, d.target
        elif dataset == "breast_cancer":
            d = load_breast_cancer(); X, y = d.data, d.target
        else:
            if experiment_type == "regression":
                X, y = make_regression(n_samples=500, n_features=10, noise=0.1, random_state=42)
            elif experiment_type == "clustering":
                X, y = make_classification(n_samples=300, n_features=2, n_classes=3,
                                           n_clusters_per_class=1, random_state=42)
            else:
                X, y = make_classification(n_samples=1000, n_features=20,
                                           n_informative=15, random_state=42)

        Xs = StandardScaler().fit_transform(X)

        if experiment_type == "clustering":
            inertias, sils, ks = [], [], range(2, 8)
            for k in ks:
                km   = KMeans(n_clusters=k, random_state=42, n_init=10)
                lbls = km.fit_predict(Xs)
                inertias.append(km.inertia_)
                sils.append(silhouette_score(Xs, lbls))
            best_k = int(list(ks)[int(np.argmax(sils))])
            km_b   = KMeans(n_clusters=best_k, random_state=42, n_init=10)
            labels = km_b.fit_predict(Xs)
            res.update({"best_k": best_k,
                        "silhouette_score": round(max(sils), 4),
                        "inertia": round(km_b.inertia_, 2)})
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot(list(ks), inertias, "bo-")
            axes[0].set_title("Elbow Curve"); axes[0].set_xlabel("k")
            Xp = PCA(n_components=2).fit_transform(Xs)
            axes[1].scatter(Xp[:,0], Xp[:,1], c=labels, cmap="viridis", alpha=0.6)
            axes[1].set_title(f"Clusters k={best_k} (PCA)")

        elif experiment_type == "regression":
            Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.2, random_state=42)
            mdl_map = {
                "random_forest"    : RandomForestRegressor(n_estimators=100, random_state=42),
                "ridge"            : Ridge(),
                "lasso"            : Lasso(),
                "linear_regression": LinearRegression(),
            }
            mdl = mdl_map.get(model_name, RandomForestRegressor(random_state=42))
            mdl.fit(Xtr, ytr)
            preds = mdl.predict(Xte)
            mse   = mean_squared_error(yte, preds)
            res.update({"mse": round(mse,4), "rmse": round(float(np.sqrt(mse)),4),
                        "r2_score": round(r2_score(yte, preds),4)})
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].scatter(yte, preds, alpha=0.5, color="steelblue")
            axes[0].plot([yte.min(), yte.max()], [yte.min(), yte.max()], "r--")
            axes[0].set_title("Actual vs Predicted")
            axes[1].hist(yte - preds, bins=30, color="coral", edgecolor="black")
            axes[1].set_title("Residuals Distribution")

        else:  # classification
            Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.2,
                                                    random_state=42, stratify=y)
            mdl_map = {
                "random_forest"    : RandomForestClassifier(n_estimators=100, random_state=42),
                "svm"              : SVC(probability=True, random_state=42),
                "logistic"         : LogisticRegression(max_iter=500, random_state=42),
                "gradient_boosting": GradientBoostingClassifier(random_state=42),
                "mlp"              : MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42),
                "knn"              : KNeighborsClassifier(n_neighbors=5),
            }
            mdl = mdl_map.get(model_name, RandomForestClassifier(random_state=42))
            mdl.fit(Xtr, ytr)
            preds     = mdl.predict(Xte)
            acc       = accuracy_score(yte, preds)
            cv_scores = cross_val_score(mdl, Xs, y, cv=5, scoring="accuracy")
            res.update({"accuracy": round(acc,4),
                        "cv_mean_accuracy": round(float(cv_scores.mean()),4),
                        "cv_std": round(float(cv_scores.std()),4),
                        "classification_report": classification_report(yte, preds)})
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            sns.heatmap(confusion_matrix(yte, preds), annot=True, fmt="d",
                        cmap="Blues", ax=axes[0])
            axes[0].set_title("Confusion Matrix")
            axes[1].bar(range(1,6), cv_scores, color="steelblue", edgecolor="black")
            axes[1].axhline(cv_scores.mean(), color="red", linestyle="--",
                            label=f"Mean={cv_scores.mean():.3f}")
            axes[1].set_title("CV Scores"); axes[1].legend()

        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format="png", dpi=120, bbox_inches="tight")
        buf.seek(0)
        res["figure_b64"] = base64.b64encode(buf.read()).decode()
        plt.close("all")
        res["status"] = "success"

    except Exception as e:
        res["status"] = "error"
        res["error"]  = str(e)
        traceback.print_exc()

    return res


# -- Tool 4: Statistical Analysis -----------------------------------------
def run_statistical_analysis(data_dict: Dict[str, List[float]]) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    try:
        for name, values in data_dict.items():
            a = np.array(values)
            out[name] = {
                "mean"    : round(float(np.mean(a)),4),
                "median"  : round(float(np.median(a)),4),
                "std"     : round(float(np.std(a)),4),
                "skewness": round(float(stats.skew(a)),4),
                "kurtosis": round(float(stats.kurtosis(a)),4),
            }
        keys = list(data_dict.keys())
        if len(keys) >= 2:
            t, p = stats.ttest_ind(np.array(data_dict[keys[0]]),
                                   np.array(data_dict[keys[1]]))
            out["t_test"] = {"t_statistic": round(float(t),4),
                              "p_value": round(float(p),6),
                              "significant": bool(p < 0.05)}
    except Exception as e:
        out["error"] = str(e)
    return out


print("All tools defined!")
print("  arXiv Search | Serper Web Search | ML Experiment | Statistical Analysis")


Groq LLM initialized: llama-3.1-8b-instant
All tools defined!
  arXiv Search | Serper Web Search | ML Experiment | Statistical Analysis


## 🤖 Cell 4: Define Agent State & All 5 Agents


In [4]:
# ============================================================
# CELL 4: Define ResearchState & All 5 Agents
# ============================================================

class ResearchState(TypedDict):
    research_topic    : str
    research_domain   : str
    experiment_type   : str
    dataset_choice    : str
    model_choice      : str
    literature_review : str
    hypotheses        : str
    experiment_results: Dict[str, Any]
    analysis_results  : str
    research_paper    : str
    current_step      : str
    logs              : List[str]
    figure_b64        : str


def add_log(state: ResearchState, msg: str) -> ResearchState:
    state["logs"].append(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")
    print(f"  >> {msg}")
    return state


# == Agent 1: Literature Review ==========================================
def literature_review_agent(state: ResearchState) -> ResearchState:
    print("\n[Agent 1] Literature Review Agent")
    state["current_step"] = "literature_review"
    topic  = state["research_topic"]
    domain = state.get("research_domain", "machine learning")

    arxiv_res = search_arxiv(f"{topic} {domain}", max_results=5)
    web_res   = search_web(f"{topic} recent research 2024", num_results=4)

    prompt = [
        SystemMessage(content=(
            "You are an expert AI research literature reviewer. "
            "Synthesize sources into a structured review: "
            "1) Overview  2) Key Methods  3) State-of-the-Art  4) Research Gaps."
        )),
        HumanMessage(content=(
            f"Topic: {topic}\nDomain: {domain}\n\n"
            f"arXiv Papers:\n{arxiv_res}\n\n"
            f"Web Research:\n{web_res}\n\n"
            "Write a structured literature review (400-500 words)."
        )),
    ]
    resp = llm.invoke(prompt)
    state["literature_review"] = resp.content
    return add_log(state, f"Literature review done ({len(resp.content)} chars)")


# == Agent 2: Hypothesis Generator =======================================
def hypothesis_generator_agent(state: ResearchState) -> ResearchState:
    print("\n[Agent 2] Hypothesis Generator Agent")
    state["current_step"] = "hypothesis_generation"

    prompt = [
        SystemMessage(content=(
            "You are a creative AI research scientist. "
            "Generate 3 testable, falsifiable hypotheses. "
            "For each: H0 (null hypothesis), H1 (alternative), expected outcome, measurement metric."
        )),
        HumanMessage(content=(
            f"Topic: {state['research_topic']}\n"
            f"Experiment Type: {state['experiment_type']}\n\n"
            f"Literature Summary:\n{state['literature_review'][:800]}\n\n"
            "Generate 3 clearly formatted hypotheses."
        )),
    ]
    resp = llm.invoke(prompt)
    state["hypotheses"] = resp.content
    return add_log(state, "3 hypotheses generated")


# == Agent 3: Experiment Agent ===========================================
def experiment_agent(state: ResearchState) -> ResearchState:
    print("\n[Agent 3] Experiment Agent")
    state["current_step"] = "running_experiments"
    exp_type = state.get("experiment_type", "classification")
    dataset  = state.get("dataset_choice",  "synthetic")
    model    = state.get("model_choice",    "random_forest")

    state = add_log(state, f"Running {exp_type} | {dataset} | {model}")
    res = run_ml_experiment(experiment_type=exp_type, dataset=dataset, model_name=model)
    state["experiment_results"] = res
    if res.get("figure_b64"):
        state["figure_b64"] = res["figure_b64"]

    res_clean = {k: v for k, v in res.items()
                 if k not in ("figure_b64", "classification_report")}
    prompt = [
        SystemMessage(content="You are an ML experiment specialist."),
        HumanMessage(content=(
            f"Experiment: type={exp_type}, dataset={dataset}, model={model}\n"
            f"Results: {json.dumps(res_clean, indent=2)}\n\n"
            "Write a brief experiment design commentary (150 words)."
        )),
    ]
    commentary = llm.invoke(prompt)
    state["experiment_results"]["commentary"] = commentary.content
    return add_log(state, f"Experiment done | status={res.get('status')}")


# == Agent 4: Result Analysis Agent ======================================
def result_analysis_agent(state: ResearchState) -> ResearchState:
    print("\n[Agent 4] Result Analysis Agent")
    state["current_step"] = "analyzing_results"

    res_clean = {k: v for k, v in state["experiment_results"].items()
                 if k not in ("figure_b64", "classification_report")}

    rng = np.random.default_rng(42)
    stat_res = run_statistical_analysis({
        "baseline_performance": rng.normal(0.75, 0.05, 20).tolist(),
        "proposed_performance": rng.normal(0.82, 0.04, 20).tolist(),
    })

    prompt = [
        SystemMessage(content=(
            "You are a rigorous AI research analyst. "
            "Analyse results, test hypotheses, draw evidence-based conclusions. "
            "Structure: 1) Summary  2) Hypothesis Testing  3) Statistical Significance  "
            "4) Limitations  5) Conclusions."
        )),
        HumanMessage(content=(
            f"Topic: {state['research_topic']}\n\n"
            f"Hypotheses:\n{state['hypotheses'][:600]}\n\n"
            f"Results:\n{json.dumps(res_clean, indent=2)}\n\n"
            f"Statistical Tests:\n{json.dumps(stat_res, indent=2)}\n\n"
            "Write a detailed analysis (350-450 words)."
        )),
    ]
    resp = llm.invoke(prompt)
    state["analysis_results"] = resp.content
    return add_log(state, "Analysis complete")


# == Agent 5: Paper Writer Agent =========================================
def paper_writer_agent(state: ResearchState) -> ResearchState:
    print("\n[Agent 5] Paper Writer Agent")
    state["current_step"] = "writing_paper"

    prompt = [
        SystemMessage(content=(
            "You are an expert academic paper writer. "
            "Write a complete IEEE-format paper: Title, Abstract, Introduction, "
            "Related Work, Methodology, Experiments, Results & Discussion, Conclusion, References."
        )),
        HumanMessage(content=(
            f"TOPIC: {state['research_topic']}\n"
            f"DOMAIN: {state.get('research_domain','Machine Learning')}\n\n"
            f"LITERATURE:\n{state['literature_review'][:600]}\n\n"
            f"HYPOTHESES:\n{state['hypotheses'][:400]}\n\n"
            f"ANALYSIS:\n{state['analysis_results'][:600]}\n\n"
            "Write a complete academic paper (700-900 words) with all sections and [N] citations."
        )),
    ]
    resp = llm.invoke(prompt)
    state["research_paper"] = resp.content
    state["current_step"]   = "completed"
    return add_log(state, "Research paper written successfully!")


print("All 5 agents defined!")


All 5 agents defined!


## 🔗 Cell 5: Build LangGraph Workflow


In [5]:
# ============================================================
# CELL 5: Build LangGraph Workflow
# ============================================================

def build_research_graph():
    wf = StateGraph(ResearchState)

    wf.add_node("literature_review",     literature_review_agent)
    wf.add_node("hypothesis_generation", hypothesis_generator_agent)
    wf.add_node("run_experiments",       experiment_agent)
    wf.add_node("analyze_results",       result_analysis_agent)
    wf.add_node("write_paper",           paper_writer_agent)

    wf.add_edge(START,                    "literature_review")
    wf.add_edge("literature_review",      "hypothesis_generation")
    wf.add_edge("hypothesis_generation",  "run_experiments")
    wf.add_edge("run_experiments",        "analyze_results")
    wf.add_edge("analyze_results",        "write_paper")
    wf.add_edge("write_paper",            END)

    return wf.compile()


research_graph = build_research_graph()
print("LangGraph workflow compiled!")
print()
print("Pipeline DAG:")
print("  START")
print("    --> Literature Review Agent")
print("    --> Hypothesis Generator Agent")
print("    --> Experiment Agent")
print("    --> Result Analysis Agent")
print("    --> Paper Writer Agent")
print("    --> END")


LangGraph workflow compiled!

Pipeline DAG:
  START
    --> Literature Review Agent
    --> Hypothesis Generator Agent
    --> Experiment Agent
    --> Result Analysis Agent
    --> Paper Writer Agent
    --> END


## 🎨 Cell 6: Launch Professional Gradio UI


In [ ]:
# ============================================================
# CELL 6: Professional Gradio UI
# ============================================================

CUSTOM_CSS = """
:root {
    --primary : #1a56db; --pri-lt: #e8f0fe; --accent: #0e9f6e;
    --bg: #f4f7ff; --surface: #ffffff; --border: #dde3f0;
    --text: #1e2a3a; --muted: #64748b;
}
body, .gradio-container {
    background: var(--bg) !important;
    font-family: 'Segoe UI', system-ui, sans-serif !important;
    color: var(--text) !important;
}
#hero {
    background: linear-gradient(135deg,#1a56db 0%,#0e9f6e 100%);
    border-radius: 16px; padding: 32px 40px; color: #fff;
    box-shadow: 0 6px 24px rgba(26,86,219,.25); margin-bottom:4px;
}
#hero h1 { font-size:1.9rem; font-weight:800; margin:0; }
#hero p  { margin:6px 0 0; opacity:.88; font-size:.93rem; }
.badge {
    display:inline-block; background:rgba(255,255,255,.2);
    border:1px solid rgba(255,255,255,.4); border-radius:20px;
    padding:4px 14px; font-size:.76rem; font-weight:700; margin:10px 3px 0;
}
#pipe {
    display:flex; gap:6px; align-items:center; flex-wrap:wrap;
    background:var(--surface); border:1px solid var(--border);
    border-radius:12px; padding:12px 18px; margin:8px 0 14px;
}
.chip {
    background:var(--pri-lt); color:var(--primary); border-radius:20px;
    padding:5px 14px; font-size:.77rem; font-weight:700; white-space:nowrap;
}
.arr { color:var(--muted); }
label { font-weight:600 !important; }
button[class*='primary'] {
    background:linear-gradient(135deg,#1a56db,#0e9f6e) !important;
    color:#fff !important; border:none !important;
    border-radius:10px !important; font-weight:700 !important;
    padding:12px 28px !important;
    box-shadow:0 4px 14px rgba(26,86,219,.3) !important;
}
button[class*='secondary'] {
    background:#fff !important; border:2px solid var(--primary) !important;
    color:var(--primary) !important; border-radius:10px !important; font-weight:600 !important;
}
.metric-row { display:flex; gap:10px; flex-wrap:wrap; margin:12px 0; }
.mchip {
    background:linear-gradient(135deg,var(--pri-lt),#fff);
    border:1px solid var(--primary); border-radius:10px;
    padding:10px 18px; min-width:110px; text-align:center;
}
.mchip .lbl { font-size:.7rem; color:var(--muted); font-weight:700; text-transform:uppercase; }
.mchip .val { font-size:1.4rem; font-weight:800; color:var(--primary); }
#logbox textarea {
    font-family:'Courier New',monospace !important; font-size:.82rem !important;
    background:#0f1729 !important; color:#a8d8a8 !important; border-radius:10px !important;
}
.tabs .tab-nav button { border-radius:8px 8px 0 0 !important; font-weight:600 !important; }
.tabs .tab-nav button.selected { background:var(--primary) !important; color:#fff !important; }
"""

HERO = """
<div id="hero">
  <h1>&#x1F52C; Autonomous AI Research Scientist</h1>
  <p>LangGraph + Groq llama-3.1-8b-instant + Serper API + arXiv</p>
  <span class="badge">&#x1F4DA; Literature Review</span>
  <span class="badge">&#x1F4A1; Hypothesis</span>
  <span class="badge">&#x1F9EA; Experiments</span>
  <span class="badge">&#x1F4CA; Analysis</span>
  <span class="badge">&#x270D; Paper Writing</span>
</div>
"""

PIPE = """
<div id="pipe">
  <span class="chip">&#x1F4DA; Literature Review</span><span class="arr">&#x2192;</span>
  <span class="chip">&#x1F4A1; Hypothesis</span><span class="arr">&#x2192;</span>
  <span class="chip">&#x1F9EA; Experiments</span><span class="arr">&#x2192;</span>
  <span class="chip">&#x1F4CA; Analysis</span><span class="arr">&#x2192;</span>
  <span class="chip">&#x270D; Paper</span>
</div>
"""

EXAMPLE_TOPICS = [
    "Transformer attention mechanisms for time series forecasting",
    "Few-shot learning with large language models",
    "Federated learning for privacy-preserving NLP",
    "Graph neural networks for drug discovery",
    "Diffusion models for image generation",
    "Reinforcement learning from human feedback (RLHF)",
    "Contrastive learning for multimodal representations",
    "Neural architecture search for edge AI devices",
]

def metric_html(res):
    keys = ["accuracy","cv_mean_accuracy","r2_score","mse","rmse","silhouette_score","best_k"]
    chips = "".join(
        f"<div class='mchip'><div class='lbl'>{k.replace('_',' ')}</div>"
        f"<div class='val'>{res[k]}</div></div>"
        for k in keys if k in res
    )
    return f"<div class='metric-row'>{chips}</div>" if chips else ""


def run_pipeline(topic, domain, exp_type, dataset, model_sel, temperature, max_tok):
    if not topic.strip():
        yield ("Please enter a research topic.", "", "", "", "", None, "")
        return

    global llm
    llm = ChatGroq(model=MODEL_NAME, temperature=temperature,
                   max_tokens=int(max_tok), groq_api_key=GROQ_API_KEY)

    state: ResearchState = {
        "research_topic": topic, "research_domain": domain,
        "experiment_type": exp_type, "dataset_choice": dataset,
        "model_choice": model_sel,
        "literature_review": "", "hypotheses": "",
        "experiment_results": {}, "analysis_results": "",
        "research_paper": "", "current_step": "starting",
        "logs": [], "figure_b64": "",
    }

    lit = hyp = exp_m = ana = pap = ""
    fig = None
    logs = []

    try:
        for event in research_graph.stream(state):
            node = list(event.keys())[0]
            ns: ResearchState = event[node]
            logs += ns.get("logs", [])
            log_str = "\n".join(logs[-30:])

            if node == "literature_review":
                lit = ns.get("literature_review", "")
            elif node == "hypothesis_generation":
                hyp = ns.get("hypotheses", "")
            elif node == "run_experiments":
                res = ns.get("experiment_results", {})
                exp_m = metric_html(res)
                if res.get("commentary"):
                    exp_m += f"<p style='margin-top:10px'>{res['commentary']}</p>"
                if ns.get("figure_b64"):
                    fig = PILImage.open(io.BytesIO(base64.b64decode(ns["figure_b64"])))
            elif node == "analyze_results":
                ana = ns.get("analysis_results", "")
            elif node == "write_paper":
                pap = ns.get("research_paper", "")

            yield (lit, hyp, exp_m, ana, pap, fig, log_str)

    except Exception as e:
        logs.append(f"ERROR: {e}\n{traceback.format_exc()}")
        yield (lit, hyp, exp_m, ana, pap, fig, "\n".join(logs))


with gr.Blocks(css=CUSTOM_CSS, title="AI Research Scientist", theme=gr.themes.Soft()) as demo:

    gr.HTML(HERO)
    gr.HTML(PIPE)

    with gr.Row(equal_height=False):

        # LEFT SIDEBAR
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("### ⚙️ Configuration")

            topic_box = gr.Textbox(label="🔍 Research Topic",
                                   placeholder="Enter your research topic...", lines=3)
            example_dd = gr.Dropdown(choices=EXAMPLE_TOPICS,
                                     label="💡 Example Topics", value=None)
            example_dd.change(fn=lambda t: t, inputs=[example_dd], outputs=[topic_box])

            gr.Markdown("---")
            domain_dd = gr.Dropdown(
                choices=["Machine Learning","Deep Learning","NLP","Computer Vision",
                         "Reinforcement Learning","Graph Neural Networks",
                         "Generative AI","Federated Learning","AutoML"],
                value="Machine Learning", label="🏛️ Research Domain")
            exp_dd = gr.Dropdown(
                choices=["classification","regression","clustering"],
                value="classification", label="🧪 Experiment Type")
            data_dd = gr.Dropdown(
                choices=["synthetic","iris","breast_cancer"],
                value="synthetic", label="📦 Dataset")
            model_dd = gr.Dropdown(
                choices=["random_forest","gradient_boosting","svm","logistic",
                         "mlp","knn","ridge","lasso","linear_regression"],
                value="random_forest", label="🤖 ML Model")

            gr.Markdown("---")
            with gr.Accordion("🎛️ LLM Settings", open=False):
                temp_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="🌡️ Temperature")
                tok_sl  = gr.Slider(512, 4096, value=2048, step=256, label="📝 Max Tokens")
                gr.Markdown(f"**Model:** `{MODEL_NAME}`")

            gr.Markdown("---")
            run_btn   = gr.Button("🚀 Launch Research Pipeline", variant="primary", size="lg")
            clear_btn = gr.Button("🗑️ Clear All", variant="secondary")

            gr.HTML("""
            <div style="margin-top:14px;padding:14px;background:#f4f7ff;
                        border:1px solid #dde3f0;border-radius:10px;font-size:.82rem">
              <div style="font-weight:700;color:#1a56db;margin-bottom:8px">&#x1F916; Active Agents</div>
              <div style="margin:4px 0">&#x1F4DA; Literature Review Agent</div>
              <div style="margin:4px 0">&#x1F4A1; Hypothesis Generator</div>
              <div style="margin:4px 0">&#x1F9EA; Experiment Agent</div>
              <div style="margin:4px 0">&#x1F4CA; Result Analysis Agent</div>
              <div style="margin:4px 0">&#x270D;&#xFE0F; Paper Writer Agent</div>
            </div>
            """)

        # RIGHT CONTENT TABS
        with gr.Column(scale=3):
            with gr.Tabs():

                with gr.TabItem("📚 Literature Review"):
                    lit_out = gr.Textbox(label="Literature Review", lines=18,
                                         show_copy_button=True,
                                         placeholder="Literature review will appear here...")

                with gr.TabItem("💡 Hypotheses"):
                    hyp_out = gr.Textbox(label="Hypotheses", lines=18,
                                         show_copy_button=True,
                                         placeholder="Hypotheses will appear here...")

                with gr.TabItem("🧪 Experiments"):
                    exp_html = gr.HTML(
                        value="<p style='color:#64748b;padding:20px'>Run the pipeline to see metrics...</p>")
                    exp_img  = gr.Image(label="📈 Visualization", type="pil", height=350)

                with gr.TabItem("📊 Analysis"):
                    ana_out = gr.Textbox(label="Result Analysis", lines=18,
                                         show_copy_button=True,
                                         placeholder="Analysis will appear here...")

                with gr.TabItem("📄 Research Paper"):
                    pap_out = gr.Textbox(label="Research Paper", lines=25,
                                         show_copy_button=True,
                                         placeholder="Full research paper will appear here...")

                with gr.TabItem("🖥️ Logs"):
                    log_out = gr.Textbox(label="Execution Log", lines=20,
                                         elem_id="logbox",
                                         placeholder="Logs will stream here...",
                                         show_copy_button=True)

    gr.HTML("""
    <div style="text-align:center;padding:16px;margin-top:14px;
                background:#fff;border-radius:12px;border:1px solid #dde3f0;
                color:#64748b;font-size:.82rem">
      <strong>Autonomous AI Research Scientist</strong> &nbsp;|&nbsp;
      LangGraph &bull; Groq llama-3.1-8b-instant &bull; Serper API &bull; arXiv
    </div>
    """)

    run_btn.click(
        fn=run_pipeline,
        inputs=[topic_box, domain_dd, exp_dd, data_dd, model_dd, temp_sl, tok_sl],
        outputs=[lit_out, hyp_out, exp_html, ana_out, pap_out, exp_img, log_out],
    )
    clear_btn.click(
        fn=lambda: ("","",
                    "<p style='color:#64748b;padding:20px'>Run the pipeline...</p>",
                    "","",None,""),
        outputs=[lit_out, hyp_out, exp_html, ana_out, pap_out, exp_img, log_out],
    )

demo.queue(max_size=5)
demo.launch(share=True, debug=True, server_name="0.0.0.0", server_port=7860)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7561d07932013a5556.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



[Agent 1] Literature Review Agent
  >> Literature review done (3866 chars)

[Agent 2] Hypothesis Generator Agent
  >> 3 hypotheses generated

[Agent 3] Experiment Agent
  >> Running classification | synthetic | random_forest
  >> Experiment done | status=success

[Agent 4] Result Analysis Agent
  >> Analysis complete

[Agent 5] Paper Writer Agent
  >> Research paper written successfully!
